In [ ]:
validar codigo:

%md
##1) Treinamento de modelo
- em Pyspark
- com pipeline
- ainda sem MLflow
%md
**Cria base de público + target + features**
#Puxa base para treinamento do modelo
df_publico = spark.sql("""SELECT cpf, --id
                                dtprocessamento, --id
                                bompagador, --target: flag = 1 se regularizou dívida
                                BCVENCIDOM0BKCPF, --feature: valor vencido no banco central
                                BCLIMCARTM6BKCPF,  --feature: limite de cartão total no banco central últ. 6m
                                CARTMDAM3BKCPF,  --feature: máximo de dias de atraso últ. 3m
                                sumVLATRZM0,  --feature: valor em atraso
                                vencidobc_sob_renda  --feature: valor vencido no banco central / renda

                            FROM data_science.base_treino_bk2
                                order by cpf, dtprocessamento
                        """)
display(df_publico)
%md
**Gera base de treino e teste**
# Dividir em 70% treinamento e 30% teste, com semente fixa para reprodutibilidade
train_df, test_df = df_publico.randomSplit([0.7, 0.3], seed=42)
print(train_df.count())
print(test_df.count())
%md
**treinamento do modelo, via pipeline**
# Em PySpark, os modelos requerem que as variáveis explicativas estejam em um único vetor de features.
# O VectorAssembler faz essa vetorização, permitindo o processamento paralelo eficiente.

from pyspark.ml.feature import VectorAssembler

# Seleciona variáveis explicativas
numeric_cols = [field for (field, dataType) in train_df.dtypes if ((dataType in  ("double",'int')) & (field != "bompagador"))]

# Cria a vetorização
vec_assembler = VectorAssembler(inputCols=numeric_cols, outputCol="features")

#Aplica a vetorização na base de treino
vec_train_df = vec_assembler.transform(train_df)
#metodologia: árvore de decisão
from pyspark.ml.classification import DecisionTreeClassifier

dt = DecisionTreeClassifier(featuresCol="features", #output da vetorização: vetor de variáveis explicativas
                            labelCol="bompagador", #target
                            maxDepth= 3) #hiperparâmetro: profundidade máxima da árvore
#Pipeline do PySpark serve para encadear etapas em um único fluxo, facilitando aplicação posterior

from pyspark.ml import Pipeline

#pipeline engloba vetorização e modelo
stages = [vec_assembler, dt]
pipeline = Pipeline(stages=stages)
#O pipeline é aplicado à base de treino, resultando em um modelo treinado (ou fitted pipeline) que pode agora ser usado para predição.
pipeline_model = pipeline.fit(train_df)
%md
**escoragem da base de teste**
display(test_df)
#pipeline já considera a vetorização e escoragem do modelo treinado anteriormente (pipeline_model), e é aplicado na base de teste
pred_df = pipeline_model.transform(test_df)
display(pred_df)

#A base resultante mantém todas as colunas originais + vetor de features + vetor de predição + coluna de predito (que por default é 1 se probabilidade do evento > 0.5)
%md
##2) Registro com MLflow
- Iniciar experimentos
- Logar parâmetros, métricas e modelo
%md
Com MLFlow é possível salvar cada rodada do modelo (.fit) como um experimento, inclusive com o notebook que criou aquele modelo e os resultados.<br>
Mesmo que altere o notebook depois, o MLflow mantém a versão exata utilizada no momento do treino.


%md
**Repete os passos acima de trazer a base de público com target e features, e quebrar em treino e teste**
df_publico = spark.sql("""SELECT cpf, --id
                                dtprocessamento, --id
                                bompagador, --target: flag = 1 se regularizou dívida
                                BCVENCIDOM0BKCPF, --feature: valor vencido no banco central
                                BCLIMCARTM6BKCPF,  --feature: limite de cartão total no banco central últ. 6m
                                CARTMDAM3BKCPF,  --feature: máximo de dias de atraso últ. 3m
                                sumVLATRZM0,  --feature: valor em atraso
                                vencidobc_sob_renda  --feature: valor vencido no banco central / renda

                            FROM data_science.base_treino_bk2
                                order by cpf, dtprocessamento
                        """)

train_df, test_df = df_publico.randomSplit([0.7, 0.3], seed=42)
print(train_df.count())
print(test_df.count())
%md
**MLFlow: utiliza o mlflow.start_run para cria experimento com treinamento via pipeline, escolhendo o o que salvar + escoragem**
import mlflow
import mlflow.spark
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.classification import DecisionTreeClassifier
from pyspark.ml import Pipeline

with mlflow.start_run(run_name='modelo_teste',
                      tags = {"modelo":"arvore",
                              "parametros":"maxDepth_3",
                              }) as run:
                      #run_name = nome do experimento, dentro do notebook. Se não definir, ele dá o nome aleatório.

    #----- Define pipeline -----
    #-- vetorização
    numeric_cols = [field for (field, dataType) in train_df.dtypes if ((dataType in  ("double",'int')) & (field != "bompagador"))]
    vec_assembler = VectorAssembler(inputCols=numeric_cols, outputCol="features")

    #-- escolha do algoritmo
    dt = DecisionTreeClassifier(featuresCol="features", labelCol="bompagador", maxDepth=3)

    #-- pipeline
    pipeline = Pipeline(stages=[vec_assembler, dt])


    #----- Treinamento -----
    #-- ajuste do modelo
    pipeline_model = pipeline.fit(train_df)

    #-- escoragem da base de treino
    pred_train_df = pipeline_model.transform(train_df)

    #-- métrica do treino
    evaluator = BinaryClassificationEvaluator(rawPredictionCol="prediction", labelCol="bompagador", metricName="areaUnderROC")
    auc_train = evaluator.evaluate(pred_train_df)


    #-----  Avaliação no teste -----
    #-- escoragem da base de teste
    pred_df = pipeline_model.transform(test_df)
    #-- métrica do teste
    auc_test = evaluator.evaluate(pred_df)


    #-----  Escolha do que registrar
    #--modelo e exemplo base treino
    mlflow.spark.log_model(pipeline_model, "model", input_example=train_df.limit(5).toPandas())

    #--target e features
    mlflow.log_param("label", "bompagador")
    mlflow.log_param("features", ", ".join(numeric_cols)) #string com nomes das features

    #--métricas
    mlflow.log_metric("auc_train", auc_train)
    mlflow.log_metric("auc_test", auc_test)
%md

- Todos os experimentos deste notebook ficam dentro da mesma "pasta", com o nome do notebook, mesmo que cluster reinicie.
<br></br>
- **Boas práticas**
  - _run_name:_ mantém nome do projeto
  - _tags:_ usar para diferenciar testes
<br></br>
- **Na aba de experimentos:**
  - Exibir as _tags_ e as _métricas principais_ para facilitar a comparação entre runs. Isso pode ser feito clicando em _"Columns"_ no canto direito e marcando as que deseja visualizar.

%md
##3) Uso produtivo
Salvar versão do modelo para reuso ou produção
<br>
<br>
As etapas são:<br>
1. Registrar o modelo _(cada registro de um mesmo modelo gera uma versão)_
2. Atribuir um estágio
    - _None: default_
    - _Stage: para testes_
    - _Production: para produtivo_
    - _Archived: desativado_
3. Carregar o modelo para usá-lo para batch inference/backtests
4. Criar um endpoint para chamada do modelo online via API, se necessário
%md
**Via User Interface**

**1. Registrar o modelo**
- Menu Machine Learning | Experiments > Selecionar Experimento > Selecionar Run > Botão "Register model"
    <br>_Pode criar um novo nome de modelo, ou escolher um nome existente e criar uma nova versão_
    <br>_Para consultar versões: Menu Machine Learning | Models > nome_do_modelo > tem todas as versões_

**2. Atribuir um estágio**
- Menu Machine Learning | Models > Selecionar modelo registrado > Selecionar versão > Escolher Stage

**3. Carregar o modelo para usá-lo para batch inference/backtests**
- Menu Machine Learning | Models > Selecionar modelo registrado > Botão "Use model for inference" > Batch inference > Adicionar a base a ser escorada

**4. Deploy via API**
- Menu Machine Learning | Models > Selecionar modelo registrado > Botão "Use model for inference" > Criar endpoint
%md
**Via código**
#--- 1. Registrar o modelo

#No final do seu experimento, ainda dentro do with mlflow.start_run(), adicionar registro no Model Registry
mlflow.register_model(
    model_uri=f"runs:/{run.info.run_id}/model", #aponta para o caminho do modelo treinado dentro do experimento
    name="nome_do_modelo" #nome do modelo registrado
)
#------ consultar versões

from mlflow.tracking import MlflowClient

client = MlflowClient()

# Lista todas as versões registradas de um modelo
for mv in client.search_model_versions("name='nome_do_modelo'"):
    print(f"Versão: {mv.version} | Status: {mv.current_stage}")
#--- 2. Atribuir estágio
from mlflow.tracking import MlflowClient

client = MlflowClient()

# Define nome do modelo e versão que você quer atualizar
model_name = "nome_do_modelo"
model_version = "1"  # exemplo: versão 1

# Atribui o estágio desejado
client.transition_model_version_stage(
    name=model_name,
    version=model_version,
    stage="Production",  # ou "Staging", "Archived", "None"
    archive_existing_versions=True  # move automaticamente a versão anterior para "Archived"
)
#--- 3. Carregar o modelo para usá-lo para batch inference/backtests

import mlflow.spark

model_name = "nome_do_modelo"
model_stage = "Staging"  # ou "Production"
    #ou model_version = "1"

model = mlflow.spark.load_model(f"models:/{model_name}/{model_stage}")
    #model = mlflow.spark.load_model(f"models:/{model_name}/{model_version}")

# Usar o modelo para escorar base nova
pred_nova = modelo_carregado.transform(nova_base)